# Experiment 2 revised pilot GPU qualification

Development-only check pinned to commit `9b3aad9ea1823e8fa4ae341bfef84b5a2c898ec7`. Verifies two visible T4 GPUs, source contracts, both full-size reference regimes, frozen-shadow actor branches, matched evaluations, and interrupted restart. The diagnostic is shortened to 20 interactions and plasticity probes are skipped. Passing this notebook will not qualify the 10,000-interaction horizon or authorize a scientific pilot.

In [ ]:
import os
os.environ.update({"JAX_PLATFORMS": "cuda,cpu", "MUJOCO_GL": "egl", "PYOPENGL_PLATFORM": "egl", "EGL_PLATFORM": "surfaceless", "MUJOCO_EGL_DEVICE_ID": "0", "WANDB_MODE": "disabled", "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1", "XLA_PYTHON_CLIENT_PREALLOCATE": "false"})
import signal
def _qualification_timeout(signum, frame):
    raise TimeoutError("Experiment 2 pilot GPU qualification exceeded its three-hour limit")
signal.signal(signal.SIGALRM, _qualification_timeout)
signal.alarm(3 * 60 * 60)
import platform
import subprocess
import sys
from pathlib import Path

SOURCE_SHA = "9b3aad9ea1823e8fa4ae341bfef84b5a2c898ec7"
SOURCE_BRANCH = "codex/exp2-online-prototype-20261107"
REPO = Path("/kaggle/temp/scaling-drl-research")
# Kaggle's internet toggle is required for both source and dependency downloads.
import socket
import time
import tempfile
import shutil

def wait_for_dns(host, attempts=6, delay=10):
    for attempt in range(attempts):
        try:
            socket.getaddrinfo(host, 443, type=socket.SOCK_STREAM)
            print("DNS available:", host)
            return
        except socket.gaierror as exc:
            print(f"DNS attempt {attempt + 1}/{attempts} failed for {host}: {exc}", flush=True)
            if attempt + 1 < attempts:
                time.sleep(delay)
    raise RuntimeError(f"Kaggle cannot resolve {host}. Check that Internet is enabled in notebook settings; if it is, retry after Kaggle networking recovers.")

for host in ("github.com", "pypi.org", "files.pythonhosted.org"):
    wait_for_dns(host)

if not REPO.exists():
    for attempt in range(3):
        with tempfile.TemporaryDirectory(prefix="exp2-clone-", dir=REPO.parent) as tmp:
            destination = Path(tmp) / "source"
            result = subprocess.run(["git", "clone", "--quiet", "--single-branch", "--branch", SOURCE_BRANCH, "https://github.com/kaves29/scaling-drl-research.git", str(destination)], text=True, capture_output=True)
            if result.returncode == 0:
                shutil.move(str(destination), str(REPO))
                break
            print(f"Git clone attempt {attempt + 1}/3 failed: {result.stderr.strip()}", flush=True)
        if attempt < 2:
            time.sleep(10)
    else:
        raise RuntimeError("Could not clone the pinned source after three attempts; check Kaggle internet access and GitHub availability.")
subprocess.run(["git", "checkout", "--quiet", "--detach", SOURCE_SHA], cwd=REPO, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == SOURCE_SHA
requirements = (REPO / "main/requirements.txt").read_text()
install_dm_control_without_optional_mazes = sys.version_info >= (3, 13)
if install_dm_control_without_optional_mazes:
    replacements = {"numpy==1.26.4": "numpy==2.1.3", "pandas==2.1.4": "pandas==2.2.3", "matplotlib==3.9.0": "matplotlib==3.9.2", "Shimmy==2.0.0": "Shimmy==1.3.0"}
    for old, new in replacements.items():
        assert requirements.count(old) == 1, f"Expected exactly one {old} pin"
        requirements = requirements.replace(old, new)
    for unused_pin in ("labmaze==1.0.6", "dm_control==1.0.38"):
        assert requirements.count(unused_pin) == 1, f"Expected exactly one {unused_pin} pin"
        requirements = requirements.replace(unused_pin + "\n", "")
runtime_requirements = Path("/kaggle/temp/exp2-runtime-requirements.txt")
runtime_requirements.write_text(requirements)
print("Python runtime:", platform.python_version(), "; installing runtime requirements:", runtime_requirements)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(runtime_requirements)], check=True)
if install_dm_control_without_optional_mazes:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "dm_control==1.0.38"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "jax-cuda13-plugin", "jax-cuda13-pjrt"], check=True, stdout=subprocess.DEVNULL)
    print("Removed Kaggle CUDA 13 JAX plugins to prevent conflicts with the pinned CUDA 12 backend.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "jax[cuda12]==0.4.34"], check=True)
import importlib.metadata as package_metadata
import flax
import gymnasium
import jax
import matplotlib
import mujoco
import numpy
# TensorFlow Probability 0.24 still uses the scalar-type predicate removed in NumPy 2.
if not hasattr(numpy, "issctype"):
    def _numpy_issctype(rep):
        return isinstance(rep, type) and issubclass(rep, numpy.generic)
    numpy.issctype = _numpy_issctype
import optax
import pandas
from dm_control import suite as dmc_suite
assert ("dog", "run") in dmc_suite.ALL_TASKS, "The selected DMC dog-run task is unavailable"
core_versions = {name: package_metadata.version(name) for name in ("jax", "numpy", "flax", "optax", "dm-control", "gymnasium", "mujoco", "pandas", "matplotlib")}
print("Core runtime imports and DMC dog-run task verified:", core_versions)
print("Pinned source and compatible runtime dependencies installed:", SOURCE_SHA)


In [ ]:
# Set rendering and JAX selection before importing either library.
os.environ.update({
    "JAX_PLATFORMS": "cuda,cpu", "MUJOCO_GL": "egl", "PYOPENGL_PLATFORM": "egl",
    "EGL_PLATFORM": "surfaceless", "MUJOCO_EGL_DEVICE_ID": "0", "WANDB_MODE": "disabled",
    "XDG_CACHE_HOME": "/kaggle/temp/.cache", "MPLCONFIGDIR": "/kaggle/temp/.mpl",
    "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1", "PYTHONDONTWRITEBYTECODE": "1", "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
})
subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], check=True)
import jax
import numpy as np
names = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).splitlines()
assert len(names) == 2 and all("T4" in name for name in names), f"Expected 2 T4 GPUs, got {names}"
gpu_devices = jax.devices("gpu")
assert jax.default_backend() == "gpu" and len(gpu_devices) == 2, gpu_devices
for device in gpu_devices:
    with jax.default_device(device):
        value = jax.device_put(np.ones((64, 64), dtype=np.float32), device)
        result_on_device = jax.jit(lambda array: array @ array)(value)
        result_on_device.block_until_ready()
        assert result_on_device.devices() == {device}, result_on_device.devices()
print("Verified JAX GPU dispatch on both T4 devices:", gpu_devices)
print("Experiment 2 training uses the default JAX GPU; this smoke does not test multi-GPU sharding.")


In [ ]:
import unittest
sys.path.insert(0, str(REPO / "main"))
loader = unittest.defaultTestLoader
suite = unittest.TestSuite(loader.loadTestsFromName(name) for name in (
    "tests.test_online_exp2", "tests.test_online_exp2_diagnostic_primitives"))
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful() and result.testsRun >= 10, f"GPU source contracts failed: {result}"


In [ ]:
import json
from unittest.mock import patch
from experiments.exp1 import compose_config
from experiments.online_exp2 import OnlineExp2
from experiments.exp12.online_exp2_diagnostic import DiagnosticActorBranch

results = {}
for regime in ("reference_small", "reference_large"):
    profile = "online_exp2_pilot_" + regime
    # Development qualification: full critic sizes, short explicitly patched schedule,
    # no plasticity assay. This is not a scientific pilot result.
    overrides = [
        "online_exp2.environments=[dog-run]", "online_exp2.training_seeds=[1]",
        "online_exp2.training_interactions=3", "online_exp2.checkpoint_steps=[2]",
        "online_exp2.probe_checkpoints=[]", "online_exp2.evaluation_episodes=1",
        "online_exp2.evaluation_mode=deterministic",
        "online_exp2.analysis_aggregation=mean_checkpoint_means",
        "online_exp2.resource_cap_gpu_hours=1",
        "online_exp2.diagnostic_horizon=20",
        "online_exp2.measurement_interactions=[0,10,20]",
        "online_exp2.diagnostic_save_every=10",
        "buffer.min_length=1", "buffer.sample_batch_size=8", "buffer.max_length=64",
        "probe.rounds=1", "probe.steps=2", "probe.pool_size=8",
        "probe.batch_size=8", "probe.eval_chunk=8",
        "logging_per_interaction_step=100", "evaluation_per_interaction_step=100",
        "num_eval_episodes=1", "actor_grad_cosine_every=100",
        "env.max_episode_steps=5", "env.action_repeat=1", "action_repeat=1",
    ]
    cfg = compose_config(str(REPO / "main/configs"), profile, overrides, allow_pilot=True)
    output = Path("/kaggle/temp") / ("revised-exp2-" + regime)
    fail_once = [regime == "reference_small"]
    original_act = DiagnosticActorBranch.act
    def interrupt_once(branch, observation, normalize=None):
        if fail_once[0] and branch.interactions == 11:
            fail_once[0] = False
            raise RuntimeError("expected qualification interruption")
        return original_act(branch, observation, normalize)
    # Validation is patched only to shorten the 10k pilot horizon in this GPU smoke.
    # All model sizes and source SAC/shadow/branch updates are real.
    with patch("experiments.online_exp2.validate", return_value=(3, [2], [])), patch.object(OnlineExp2, "_probe", lambda self, step: None):
        if regime == "reference_small":
            with patch.object(DiagnosticActorBranch, "act", interrupt_once):
                try:
                    OnlineExp2(cfg, str(output)).run()
                except RuntimeError as exc:
                    assert str(exc) == "expected qualification interruption", exc
                else:
                    raise AssertionError("interruption did not occur")
        runner = OnlineExp2(cfg, str(output))
        runner.run()
        expected_arch = (2, 512) if regime == "reference_small" else (4, 1536)
        assert (runner.cfg.agent.critic_num_blocks, runner.cfg.agent.critic_hidden_dim) == expected_arch
        diagnostics = [row for row in runner.rows if row.get("type") == "pilot_diagnostic"]
        evaluations = [row for row in runner.rows if row.get("type") == "pilot_evaluation"]
        assert len(diagnostics) == 1 and len(evaluations) == 2
        assert {row["diagnostic_interactions"] for row in evaluations} == {10, 20}
        assert diagnostics[0]["branch_counters"]["small"]["interactions"] == 20
        assert diagnostics[0]["branch_counters"]["large"]["interactions"] == 20
        assert all(np.isfinite(row["advantage_large"]) for row in evaluations)
        results[regime] = {"analysis": json.loads((output / "analysis.json").read_text()),
                           "diagnostic": diagnostics[0], "evaluations": evaluations}
        print("PASS full-size GPU branch:", regime, "reference", expected_arch,
              "optimizer updates", diagnostics[0]["branch_counters"], flush=True)
print("PASS revised Experiment 2 GPU qualification; development schedule only", flush=True)

summary = {name: {"analysis": value["analysis"],
                  "branch_counters": value["diagnostic"]["branch_counters"]}
           for name, value in results.items()}
Path("/kaggle/working/exp2_gpu_qualification_summary.json").write_text(
    json.dumps(summary, indent=2, sort_keys=True))
print("Wrote compact qualification summary; source and checkpoints stayed in /kaggle/temp")
